# MLE for the binomial, exponential and normal distributions
Each closed-form answer of the Note checked two ways: a grid search over the parameter, and SciPy's fitting.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats, optimize

# one seeded random generator for the whole notebook
rng = np.random.default_rng(0)

## 1. Binomial: 4 of 7 prefer orange

In [ ]:
n, x = 7, 4
p_grid = np.linspace(0.001, 0.999, 9981)
# likelihood of every p on the grid, data fixed
L = stats.binom(n, p_grid).pmf(x)
print("grid search:", p_grid[L.argmax()].round(4), " formula x/n:", round(x / n, 4))
# the same by minimising the negative log-likelihood
res = optimize.minimize_scalar(lambda p: -stats.binom(n, p).logpmf(x), bounds=(0.001, 0.999), method="bounded")
print("numerical:", res.x.round(4))

The log-likelihood peaks at the same p (Figure 2).

In [ ]:
import numpy as np
from scipy import stats
p = np.linspace(0.001, 0.999, 9999)
L = stats.binom(7, p).pmf(4)
print("argmax L:", p[L.argmax()].round(3), " argmax log L:", p[np.log(L).argmax()].round(3))
print("log L at 0.25, 0.5, 0.57:", np.log(stats.binom(7, [0.25, 0.5, 0.57]).pmf(4)).round(2))

More data, same share: the curve gets narrower. Each curve is divided by its peak so both reach 1.

In [ ]:
fig = go.Figure()
for nn, xx in ((7, 4), (70, 40)):
    Lp = stats.binom(nn, p_grid).pmf(xx)
    fig.add_trace(go.Scatter(x=p_grid, y=Lp / Lp.max(), mode="lines", name=f"{xx} of {nn}"))
fig.update_layout(template="simple_white", xaxis_title="p", yaxis_title="likelihood / its maximum", height=380)
fig.show()

## 2. Exponential: waits of 2, 2.5 and 1.5 seconds

In [ ]:
waits = np.array([2, 2.5, 1.5])
lam_grid = np.linspace(0.01, 3, 3000)
# log-likelihood n log(lambda) - lambda * sum(x)
ll = len(waits) * np.log(lam_grid) - lam_grid * waits.sum()
print("grid search:", lam_grid[ll.argmax()].round(3), " formula n/sum:", len(waits) / waits.sum())
# SciPy fits the scale 1/lambda; floc=0 keeps the curve starting at 0
loc, scale = stats.expon.fit(waits, floc=0)
print("scipy: scale =", scale, " so lambda =", 1 / scale)
# without floc=0 SciPy also fits the shift loc: the curve then starts at the smallest wait
print("scipy without floc (loc, scale):", stats.expon.fit(waits))

## 3. Normal: the five mice

In [ ]:
mice = np.array([29, 31, 32, 33, 35.0])
print("formula: mean =", mice.mean(), " std (divide by n) =", mice.std())
print("scipy norm.fit:", stats.norm.fit(mice))
# grid search over both parameters
mu, sg = np.meshgrid(np.linspace(30, 34, 401), np.linspace(1, 3.5, 251))
LL = stats.norm(mu[..., None], sg[..., None]).logpdf(mice).sum(axis=-1)
i, j = np.unravel_index(LL.argmax(), LL.shape)
print("grid search: mu =", mu[i, j].round(2), " sigma =", sg[i, j].round(2), " log-likelihood =", LL[i, j].round(2))

## 4. One point cannot give sigma: the likelihood grows without limit

In [ ]:
for s in (2, 0.5, 0.1, 0.01):
    print(f"sigma = {s}: likelihood of one mouse at its own mean = {stats.norm(32, s).pdf(32):.2f}")

## 5. Divide by n or by n - 1: simulation
100,000 samples of 5 mice from N(32, 2^2), whose true variance is 4.

In [ ]:
samples = rng.normal(32, 2, size=(100_000, 5))
print("average MLE variance (divide by n):    ", samples.var(axis=1).mean().round(3), "  theory 3.2")
print("average sample variance (divide by n-1):", samples.var(axis=1, ddof=1).mean().round(3), "  theory 4")